# Evaluate count-scale reconstruction

This notebook makes the transformation from rate and SF explicit, checks
the tile-level output produced by notebook 00, and exposes the full HEST
evaluation API for rate-only, predicted-SF and oracle-SF comparisons.


In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
from IPython.display import display


def find_project_root(start: Path) -> Path:
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / "pyproject.toml").exists():
            return candidate
    raise RuntimeError("Run this notebook from inside the HistoOmniST repository.")


ROOT = find_project_root(Path.cwd())
sys.path.insert(0, str(ROOT / "src"))

from histoomnist.inference.count_scale import mean_one_sf_from_log, reconstruct_count_scale
from histoomnist.eval.evaluate_combined import evaluate
from histoomnist.utils.config import load_config


## 1. Verify the reconstruction algebra on a small example


In [ ]:
rate = np.array([[2.0, 4.0], [3.0, 5.0]], dtype=np.float32)
rate_log1p = np.log1p(rate)
raw_log_sf = np.log(np.array([0.5, 1.5], dtype=np.float32))
normalized_log_sf, sf = mean_one_sf_from_log(raw_log_sf)
count, count_log1p = reconstruct_count_scale(rate_log1p, sf)

expected_count = rate * sf[:, None]
np.testing.assert_allclose(count, expected_count, rtol=1e-6)
np.testing.assert_allclose(count_log1p, np.log1p(expected_count), rtol=1e-6)
assert np.isclose(sf.mean(), 1.0)

display(pd.DataFrame({"raw_log_sf": raw_log_sf, "normalized_log_sf": normalized_log_sf, "SF": sf}))
print("Reconstructed count matrix:\n", count)


## 2. Audit the real Rep1 tile predictions from notebook 00


In [ ]:
quickstart_predictions = ROOT / "outputs/example_breast_xenium/rep1_tile_predictions.csv"
if not quickstart_predictions.exists():
    print("Run notebook 00 first to create", quickstart_predictions)
    rep1 = None
else:
    rep1 = pd.read_csv(quickstart_predictions)
    genes_to_check = ["EPCAM", "CD3D", "MKI67", "COL1A1"]
    for gene in genes_to_check:
        rate_values = np.clip(np.expm1(rep1[f"rate_log1p_{gene}"]), 0.0, None)
        expected_count = rate_values * rep1["pred_sf"].to_numpy()
        np.testing.assert_allclose(rep1[f"count_{gene}"], expected_count, rtol=1e-5, atol=1e-6)
        np.testing.assert_allclose(rep1[f"count_log1p_{gene}"], np.log1p(expected_count), rtol=1e-5, atol=1e-6)
    print(f"Verified rate * SF reconstruction for {len(genes_to_check)} genes across {len(rep1):,} tiles.")
    display(
        rep1[["pred_sf", "rate_log1p_EPCAM", "count_EPCAM", "count_log1p_EPCAM"]]
        .describe()
        .T
    )


## 3. Run the full held-out HEST comparison

This API keeps the rate model fixed and evaluates four quantities against
measured HEST targets: rate, count reconstructed without SF, count
reconstructed with predicted SF, and count reconstructed with oracle
measured SF. The cell remains guarded because it needs all held-out HEST
processed arrays and evaluates 16,942 genes.


In [ ]:
RUN_FULL_HEST_EVALUATION = False

if RUN_FULL_HEST_EVALUATION:
    rate_config = load_config(ROOT / "configs/hest1k_human_visium_expression_highconf_symbol95.yaml")
    sf_config = load_config(ROOT / "configs/hest1k_human_visium_sf_context_distribution_light.yaml")
    rate_config["device"] = "auto"
    sf_config["device"] = "auto"

    metrics = evaluate(
        sf_config=sf_config,
        expression_config=rate_config,
        sf_checkpoint=ROOT / "checkpoints/hest1k_human_visium_sf/context_distribution_light_hipt256_leave_slide_out/best.pt",
        expression_checkpoint=ROOT / "checkpoints/hest1k_human_visium_expression/highconf_symbol95_rate/best.pt",
        split_names=["test"],
        out_json=ROOT / "outputs/hest_count_scale_evaluation/test_metrics.json",
    )
    display(pd.Series(metrics, name="value").to_frame())
else:
    print("Full HEST evaluation not started. Set RUN_FULL_HEST_EVALUATION=True when processed assets are present.")
